In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from pyspark.sql import functions as F

In [2]:
# Start the Spark Session
spark = SparkSession.builder \
    .appName("TaxiTrajectoryAnalysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

# Schema definition
schema = StructType([
    StructField("trip_id", StringType(), True),
    StructField("taxi_id", LongType(), True),
    StructField("timestamp", LongType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True)
])

# Read csv from local
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("/workspace/data/gps_cleaned.csv")

# Print Schema 
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/20 21:53:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/20 21:53:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)



In [3]:
trip_counts = df.groupBy("trip_id").count()

# Filter-out outliers
clean_trips = trip_counts.filter("count < 10000").drop("count")

df_clean = df.join(clean_trips, "trip_id", "inner")

# Aggregate paths by row
df_paths = df_clean.groupBy("trip_id").agg(
    F.collect_list(F.struct("timestamp", "latitude", "longitude")).alias("trajectory")
)


In [4]:
# Simple helper functions
import math
import heapq

def distance(lat1, lon1, lat2, lon2):
    """Calculate distance between two GPS points in meters"""
    R = 6371000  # Earth radius in meters
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

print("Helper functions loaded")

Helper functions loaded


In [5]:
# Build simple road network from GPS data

# Create segments from trajectories (stream with toLocalIterator + cap)
segments = []
for row in df_paths.limit(2000).toLocalIterator():  # reverted to 1000
    trajectory = row.trajectory
    for i in range(len(trajectory) - 1):
        p1 = trajectory[i]
        p2 = trajectory[i + 1]
        segments.append((p1.latitude, p1.longitude, p2.latitude, p2.longitude))

print(f"Created {len(segments)} segments from GPS data")

# Build graph: nodes are GPS coordinates, edges are segments
graph = {}
nodes = {}
node_id = 0

def get_node_id(lat, lon):
    global node_id
    # Round coordinates to create grid (reverted to 3 decimals)
    lat_round = round(lat, 3)
    lon_round = round(lon, 3)
    key = (lat_round, lon_round)
    
    if key not in nodes:
        nodes[key] = node_id
        graph[node_id] = {}
        node_id += 1
    return nodes[key]

# Add edges to graph
for lat1, lon1, lat2, lon2 in segments:
    node1 = get_node_id(lat1, lon1)
    node2 = get_node_id(lat2, lon2)
    
    if node1 != node2:
        dist = distance(lat1, lon1, lat2, lon2)
        graph[node1][node2] = dist
        graph[node2][node1] = dist  # bidirectional

print(f"Built graph with {len(graph)} nodes")

# Create reverse mapping: node_id -> (lat, lon)
id_to_coord = {v: k for k, v in nodes.items()}

[Stage 4:======================================================>  (26 + 1) / 27]

Created 86434 segments from GPS data
Built graph with 7146 nodes


In [6]:
# Dijkstra algorithm implementation

def dijkstra(start_node, end_node):
    """Find shortest path between two nodes"""
    if start_node not in graph or end_node not in graph:
        return [], float('inf')
    
    # Priority queue: (distance, node, path)
    pq = [(0, start_node, [start_node])]
    visited = set()
    
    while pq:
        current_dist, current_node, path = heapq.heappop(pq)
        
        if current_node in visited:
            continue
        
        visited.add(current_node)
        
        if current_node == end_node:
            return path, current_dist
        
        # Check neighbors
        for neighbor, edge_dist in graph[current_node].items():
            if neighbor not in visited:
                new_dist = current_dist + edge_dist
                new_path = path + [neighbor]
                heapq.heappush(pq, (new_dist, neighbor, new_path))
    
    return [], float('inf')

def find_nearest_node(lat, lon):
    """Find nearest node to given coordinates"""
    min_dist = float('inf')
    nearest = None
    
    for coord, node_id in nodes.items():
        d = distance(lat, lon, coord[0], coord[1])
        if d < min_dist:
            min_dist = d
            nearest = node_id
    
    return nearest

print("Dijkstra algorithm ready")

Dijkstra algorithm ready


In [7]:
# Route calculation between two lat/lon points

def calculate_route(start_lat, start_lon, end_lat, end_lon):
    """Calculate shortest route between two GPS coordinates"""
    print(f"Route from ({start_lat:.4f}, {start_lon:.4f}) to ({end_lat:.4f}, {end_lon:.4f})")
    
    # Find nearest nodes
    start_node = find_nearest_node(start_lat, start_lon)
    end_node = find_nearest_node(end_lat, end_lon)
    
    if start_node is None or end_node is None:
        return {"success": False, "message": "No nearby nodes found"}
    
    # Calculate path using Dijkstra
    path_nodes, total_distance = dijkstra(start_node, end_node)
    
    if not path_nodes:
        return {"success": False, "message": "No route found"}
    
    # Convert to coordinates
    path_coords = []
    for node in path_nodes:
        lat, lon = id_to_coord[node]
        path_coords.append({"lat": lat, "lon": lon})
    
    return {
        "success": True,
        "path": path_coords,
        "distance_meters": total_distance,
        "distance_km": total_distance / 1000,
        "num_points": len(path_coords)
    }

print("Route calculation function ready")

Route calculation function ready


In [8]:
import random

trip_id = 0;

# Ensure the graph exists (run the graph-building cell first)
if not id_to_coord:
    raise ValueError("Graph not built yet. Run the previous cells first.")

# Pick two random distinct node IDs from the graph
node_ids = list(id_to_coord.keys())
start_node = random.choice(node_ids)
end_node = random.choice(node_ids)
while end_node == start_node:
    end_node = random.choice(node_ids)

# Map nodes to lat/lon
start_lat, start_lon = id_to_coord[start_node]
end_lat, end_lon   = id_to_coord[end_node]

print("Random start/end (from graph nodes):")
print(f"Start: ({start_lat:.5f}, {start_lon:.5f})  -> node {start_node}")
print(f"End:   ({end_lat:.5f}, {end_lon:.5f})  -> node {end_node}")

# Compute route
res = calculate_route(start_lat, start_lon, end_lat, end_lon)
print("\nResult:", res["success"])
if res["success"]:
    print(f"Distance: {res['distance_meters']:.0f} m ({res['distance_km']:.2f} km)")
    print(f"Points: {res['num_points']}")
    print("\nFull path coordinates:")
    for i, p in enumerate(res["path"], 1):
        print(f"{i:3d}: {p['lat']:.6f}, {p['lon']:.6f}")
else:
    print("Message:", res["message"])

Random start/end (from graph nodes):
Start: (41.15300, -8.67000)  -> node 2863
End:   (41.15900, -8.60500)  -> node 133
Route from (41.1530, -8.6700) to (41.1590, -8.6050)

Result: True
Distance: 4196 m (4.20 km)
Points: 43

Full path coordinates:
  1: 41.153000, -8.670000
  2: 41.152000, -8.668000
  3: 41.151000, -8.666000
  4: 41.151000, -8.663000
  5: 41.150000, -8.662000
  6: 41.150000, -8.661000
  7: 41.151000, -8.661000
  8: 41.151000, -8.660000
  9: 41.151000, -8.657000
 10: 41.151000, -8.656000
 11: 41.152000, -8.654000
 12: 41.153000, -8.651000
 13: 41.154000, -8.650000
 14: 41.154000, -8.649000
 15: 41.154000, -8.648000
 16: 41.154000, -8.645000
 17: 41.154000, -8.642000
 18: 41.153000, -8.639000
 19: 41.153000, -8.637000
 20: 41.153000, -8.636000
 21: 41.153000, -8.634000
 22: 41.153000, -8.633000
 23: 41.153000, -8.632000
 24: 41.152000, -8.631000
 25: 41.153000, -8.630000
 26: 41.153000, -8.629000
 27: 41.153000, -8.628000
 28: 41.153000, -8.627000
 29: 41.154000, -8.62700

In [9]:
import psycopg2
import json

# Connection to Postgres
conn = psycopg2.connect(
    host="postgres",   # your service name in docker-compose
    port=5432,
    database="taxi_db",
    user="taxi",
    password="taxi"
)
cur = conn.cursor()

# Example route data from your Dijkstra result
route_data = res  # the dict from calculate_route 
trip_id = "trip_example";
insert_query = """
INSERT INTO taxi_routes 
(trip_id, start_lat, start_lon, end_lat, end_lon, path, distance_m) 
VALUES (%s, %s, %s, %s, %s, %s, %s)
RETURNING route_id;
"""

cur.execute(
    insert_query,
    (
        trip_id,
        start_lat,
        start_lon,
        end_lat,
        end_lon,
        json.dumps(route_data["path"]),  # convert list of dicts to JSON string
        route_data["distance_meters"]
    )
)

route_id = cur.fetchone()[0]
conn.commit()
cur.close()
conn.close()

print(f"Route saved with route_id={route_id}")


Route saved with route_id=1
